# Deploy GLM-5.2-NVFP4 on an Amazon SageMaker AI Inference Component (ml.g7e.48xlarge)

This example deploys [nvidia/GLM-5.2-NVFP4](https://huggingface.co/nvidia/GLM-5.2-NVFP4) — the NVFP4 (4-bit) quantization of Z.ai's 753B-parameter GLM-5.2 MoE — on a single `ml.g7e.48xlarge` (8x NVIDIA Blackwell, 8x96 GB) using **SageMaker Inference Components** and the **vLLM Deep Learning Container**, with `boto3`.

Why NVFP4 + g7e: the bf16 weights (~1.5 TB) and even the FP8 release (~753 GB) exceed the instance's 768 GB total GPU memory. The NVFP4 checkpoint is ~464 GB on disk (~58 GB per GPU at TP=8), leaving comfortable headroom for KV cache.

Why an Inference Component: ICs decouple the model from the endpoint and make GPU/CPU/memory allocation explicit — and multiple ICs can later share the same instance. The endpoint is created *without* a model; the model is attached as a component with an explicit accelerator claim.

> **Container version matters.** vLLM **0.26.0** is required. On the 0.25.1 DLC this model fails at runtime with `TypeError: trtllm_batch_decode_with_kv_cache_mla() got an unexpected keyword argument 'kv_scale_format'` — a flashinfer signature mismatch on the MLA decode path. 0.26.0 ships a matched Blackwell sparse-MLA backend (`FLASHINFER_MLA_SPARSE_SM120`) and serves this architecture out of the box.

In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts boto3

In [ ]:
import sys
import time
import re
import json
import boto3
from IPython.display import display, Markdown, clear_output

boto_session = boto3.Session()
region = boto_session.region_name

sm = boto3.client("sagemaker")  # client to intreract with SageMaker
sm_runtime = boto3.client("sagemaker-runtime")  # client to intreract with SageMaker Endpoints

In [ ]:
#
# Helper functions to remove dependency on SageMaker Python SDK
#
def get_sagemaker_role():
    arn = boto3.client("sts").get_caller_identity()["Arn"]
    return re.sub(r"^(.+)sts::(\d+):assumed-role/(.+?)/.*$", r"\1iam::\2:role/\3", arn)


def _wait_for_resource(describe_fn, name_key, status_key, label, name, sleep_time=60):
    """Poll a SageMaker resource until it leaves 'Creating' or "Updating" state."""
    progress = ""
    while True:
        status = describe_fn(**{name_key: name})[status_key]
        if status not in ("Creating", "Updating"):
            break
        progress += "."
        clear_output(wait=True)
        print(f"Waiting for '{name}': {progress}")
        time.sleep(sleep_time)
    print(f"{label}: '{name}', Status: '{status}'")


def wait_for_endpoint(endpoint_name: str, sleep_time: int = 60):
    _wait_for_resource(
        sm.describe_endpoint, "EndpointName", "EndpointStatus",
        "Endpoint", endpoint_name, sleep_time,
    )


def wait_for_ic(ic_name: str, sleep_time: int = 60):
    _wait_for_resource(
        sm.describe_inference_component, "InferenceComponentName", "InferenceComponentStatus",
        "IC", ic_name, sleep_time,
    )

In [ ]:
#
# Overwrite with your role ARN if you are running this notebook outside of SageMaker Studio
#
role = None

if role is None:
    role = get_sagemaker_role()
print(role)

## Configuration

One `ml.g7e.48xlarge` hosts the model at tensor parallelism 8. The 1M-token native context is capped to 16,384 for serving — raise it if your workload needs longer prompts and you have KV headroom.

In [ ]:
instance = {"type": "ml.g7e.48xlarge", "num_gpu": 8}
model_id = "nvidia/GLM-5.2-NVFP4"
model_name = f"model-{time.strftime('%y%m%d-%H%M%S')}"
endpoint_name = model_name
endpoint_config_name = model_name
ic_name = f"{model_name}-ic"
variant_name = "v1"
# ~464 GB of weights download from the HF Hub at startup — allow generous timeouts
download_timeout = 3600
startup_timeout = 3600

## Container

SageMaker [vLLM DLC](https://aws.github.io/deep-learning-containers/vllm/) **0.26.0** — the minimum version that serves GLM-5.2's `glm_moe_dsa` architecture (sparse MLA attention) on Blackwell. The model card's parsers for tool-calling and reasoning are configured via `SM_VLLM_*` environment variables, which the DLC maps to vLLM CLI flags.

In [ ]:
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/vllm:0.26.0-gpu-py312-cu130-ubuntu22.04-sagemaker"

common_env = {
    "HF_TOKEN": "<YOUR_TOKEN>",  # not required for this public checkpoint, but raises HF rate limits
    "SM_NUM_GPUS": json.dumps(instance["num_gpu"]),
}

vllm_env = {
    "SM_VLLM_MODEL": model_id,
    "SM_VLLM_TENSOR_PARALLEL_SIZE": json.dumps(instance["num_gpu"]),
    "SM_VLLM_MAX_MODEL_LEN": "16384",
    "SM_VLLM_GPU_MEMORY_UTILIZATION": "0.92",
    "SM_VLLM_TOOL_CALL_PARSER": "glm47",
    "SM_VLLM_ENABLE_AUTO_TOOL_CHOICE": "true",
    "SM_VLLM_REASONING_PARSER": "glm45",
    # Uncomment for balanced performance across the 256 experts
    #"SM_VLLM_ENABLE_EXPERT_PARALLEL": "true",
}
env = common_env | vllm_env

reasoning_keyword = "reasoning"

## Deployment

Three steps, in order:
1. **Endpoint config + endpoint** — note the variant carries *no* `ModelName`; it only provisions the instance. `ManagedInstanceScaling` and a routing strategy are required for IC-hosting endpoints.
2. **Model** — the vLLM container plus environment.
3. **Inference component** — attaches the model to the endpoint with an explicit claim on all 8 accelerators.

In [ ]:
_ = sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ExecutionRoleArn=role,
    ProductionVariants=[
        {
            "VariantName": variant_name,
            "InstanceType": instance["type"],
            "InitialInstanceCount": 1,
            "ManagedInstanceScaling": {"Status": "ENABLED", "MinInstanceCount": 1, "MaxInstanceCount": 1},
            "RoutingConfig": {"RoutingStrategy": "LEAST_OUTSTANDING_REQUESTS"},
        },
    ],
)

_ = sm.create_endpoint(EndpointName=endpoint_name,
                       EndpointConfigName=endpoint_config_name)

wait_for_endpoint(endpoint_name)

In [ ]:
_ = sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": inference_image,
        "Environment": env,
    },
)

In [ ]:
_ = sm.create_inference_component(
    InferenceComponentName=ic_name,
    EndpointName=endpoint_name,
    VariantName=variant_name,
    Specification={
        "ModelName": model_name,
        "StartupParameters": {
            "ModelDataDownloadTimeoutInSeconds": download_timeout,
            "ContainerStartupHealthCheckTimeoutInSeconds": startup_timeout,
        },
        "ComputeResourceRequirements": {
            "NumberOfAcceleratorDevicesRequired": instance["num_gpu"],
            "NumberOfCpuCoresRequired": 100,
            "MinMemoryRequiredInMb": 1000000,
        },
    },
    RuntimeConfig={"CopyCount": 1},
)

wait_for_ic(ic_name)

## Inference examples

Requests target the endpoint **plus** the inference component (`InferenceComponentName`). The container serves the OpenAI chat-completions schema.

### Text inference

In [ ]:
payload = {
    "messages": [
        {"role": "user", "content": "Who are you?"}
    ],
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 InferenceComponentName=ic_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()
print(f"\u2705 Response time: {end_time-start_time:.2f}s\n")

reasoning = response["choices"][0]["message"].get(reasoning_keyword, None)
content = response["choices"][0]["message"].get('content', None)
if reasoning:
    display(Markdown('### Reasoning:\n---'))
    display(Markdown(reasoning))
    sys.stdout.flush()
if content:
    display(Markdown('### Content:\n---'))
    display(Markdown(content))
    sys.stdout.flush()

### Text generation, no reasoning

In [ ]:
payload = {
    "messages": [
        {"role": "user", "content": "Write a haiku about mixture-of-experts models."}
    ],
    "chat_template_kwargs": {"enable_thinking": False},
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 InferenceComponentName=ic_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()
print(f"\u2705 Response time: {end_time-start_time:.2f}s\n")

display(Markdown(response["choices"][0]["message"]["content"]))

## Cleanup

Delete the inference component first — the endpoint cannot be deleted while a component is attached. The g7e.48xlarge bills until the endpoint is gone.

In [ ]:
_ = sm.delete_inference_component(InferenceComponentName=ic_name)

# wait for the IC to detach before deleting the endpoint
while True:
    try:
        sm.describe_inference_component(InferenceComponentName=ic_name)
        time.sleep(20)
    except sm.exceptions.ClientError:
        break

_ = sm.delete_endpoint(EndpointName=endpoint_name)
_ = sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
_ = sm.delete_model(ModelName=model_name)